# Intención de compra · desarrollo en Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allisonloangorayo-sudo/intencion-compra-streamlit/blob/main/notebooks/01_desarrollo_colab.ipynb)

Ruta A (supervisada). Este cuaderno cubre las secciones 4 a 16 de la guía: datos, entrenamiento, evaluación y prueba de la app Streamlit.
Docker se construye después en la VM de Google Cloud (ver `docs/GUIA_PASO_A_PASO.md`).

Ejecute las celdas en orden.

## 1. Clonar el proyecto e instalar dependencias

In [ ]:
%cd /content
!rm -rf intencion-compra-streamlit
!git clone -q https://github.com/allisonloangorayo-sudo/intencion-compra-streamlit.git
%cd /content/intencion-compra-streamlit
!ls -la

In [ ]:
# Mismas versiones que la imagen Docker (model.joblib depende de la versión de scikit-learn)
!pip -q install -r requirements-dev.txt

> Si después de instalar aparece un error de importación, use **Entorno de ejecución → Reiniciar sesión** y vuelva a ejecutar desde la celda siguiente (sin volver a clonar).

## 2. Revisión mínima del dataset (sección 6 de la guía)

In [ ]:
%cd /content/intencion-compra-streamlit
import pandas as pd

df = pd.read_csv("data/online_shoppers_intention.csv")
print(df.shape)
display(df.head())
display(df.dtypes)
display(df.isna().sum().sort_values(ascending=False).head(15))
display(df.describe(include="all").T)

In [ ]:
# Balance de la variable objetivo
display(df["Revenue"].value_counts(normalize=True).rename("proporción"))

# Comportamiento típico de quien compra vs. quien no compra
num = ["ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues"]
display(df.groupby("Revenue")[num].median().round(3))

**Respuestas a las preguntas de la guía**

- **¿Qué representa una fila?** Una sesión de navegación de un usuario en una tienda online (12 330 sesiones en un año).
- **¿Objetivo del análisis?** Predecir si la sesión termina en compra (`Revenue`): clasificación binaria.
- **¿Qué variables entran?** Conteos y duraciones por tipo de página, tasas de rebote y salida, valor de página, cercanía a fecha especial, mes, tipo de visitante y fin de semana.
- **¿Qué variables se excluyen?** `OperatingSystems`, `Browser`, `Region`, `TrafficType`: códigos anónimos sin etiqueta, imposibles de llenar con sentido en la app.
- **¿Datos faltantes?** No hay. El pipeline igual incluye imputación por si un caso nuevo llega incompleto.
- **¿Numéricas y categóricas?** Sí: 10 numéricas (muy asimétricas → `log1p`) y 3 categóricas (one-hot).
- **Desbalance:** solo 15,5 % compra → se evalúa con recall, F1 de la clase compra y ROC-AUC, y se usa `class_weight="balanced"`.

## 3. Entrenar con `train.py` (sección 11)

In [ ]:
!python train.py

In [ ]:
import json
metrics = json.load(open("metrics.json"))
metrics

**Lectura de resultados.** El random forest detecta cerca de 3 de cada 4 sesiones que terminan en compra (recall ≈ 0,77) con ROC-AUC ≈ 0,93.
Sin `PageValues` el ROC-AUC cae a ≈ 0,76: es la variable más informativa, y por eso el README discute por qué es válida en el momento de predecir.

## 4. Comprobar que el artefacto carga sin reentrenar

In [ ]:
import joblib

artifact = joblib.load("model.joblib")
print(artifact["model_name"], "| tarea:", artifact["task"])
print("Variables:", artifact["features"])

caso = pd.DataFrame([{
    "Administrative": 3, "Administrative_Duration": 80.0,
    "Informational": 1, "Informational_Duration": 30.0,
    "ProductRelated": 35, "ProductRelated_Duration": 1400.0,
    "BounceRates": 0.0, "ExitRates": 0.012, "PageValues": 40.0,
    "SpecialDay": 0.0, "Month": "Nov", "VisitorType": "New_Visitor", "Weekend": "False",
}], columns=artifact["features"])
print("Probabilidad de compra:", round(artifact["model"].predict_proba(caso)[0, 1], 3))

## 5. Probar Streamlit en Colab (sección 15)

In [ ]:
!pkill -f "streamlit run" || true
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true > /content/streamlit.log 2>&1 &
import time; time.sleep(6)
!cat /content/streamlit.log
!curl -s localhost:8501/_stcore/health && echo "  <- Streamlit responde"

In [ ]:
# Túnel temporal con Cloudflare (no pide contraseña).
# localtunnel, el que sugiere la guía, dejaba la página en blanco con esta app.
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

In [ ]:
# Abra el enlace https://...trycloudflare.com que aparece. Detenga la celda (⏹) cuando termine de probar.
!./cloudflared tunnel --url http://localhost:8501 2>&1 | grep --line-buffered -o "https://[a-z0-9-]*\.trycloudflare\.com"

Pruebe las fichas Visita típica, Alta intención y Rebote rápido, y un caso extremo (valor de página = 300).

Si la red bloquea el túnel, basta con que el log diga *You can now view your Streamlit app*; continúe con GCP.

## 6. Pruebas mínimas antes de Docker (sección 16)

In [ ]:
!python -m pytest -q -s

Checklist:
1. `train.py` genera `model.joblib` ✔ (sección 3)
2. Se probaron 3 casos ✔ (pytest + selector de ejemplos en la app)
3. Columnas de entrada = columnas de entrenamiento ✔ (`artifact["features"]`)
4. Probar a mano valores extremos (p. ej. `ExitRates = 0.2`, `PageValues = 300`)
5. Reiniciar Streamlit y verificar que carga sin reentrenar ✔ (la app solo hace `joblib.load`)
6. Ejemplo de entrada/salida documentado en el README ✔

## 7. Siguiente paso
Seguir `docs/GUIA_PASO_A_PASO.md`, fase 3: crear la VM en Google Cloud.